# 01 · From GEOtop to GEOtopPY

For users of GEOtop 3.0 (C++). This notebook does not explain GEOtop: it runs
a case you could run with the C++ model and shows what is the same and what
changes in practice.

**What stays the same**

- The case directory: `geotop.inpts` and the same input files (meteo, soil,
  points, horizon, vegetation). Existing 1D cases run without conversion.
- The keywords and their defaults, generated from the v3.0 sources.
- The output files, their columns and their format, so existing
  post-processing scripts keep working.
- The results, within 1e-5 (absolute or relative) on the 13 official 1D cases.

**What changes**

| | GEOtop C++ | GEOtopPY |
|---|---|---|
| Install | compile | `pip install -e .`, standard library only |
| Run | `geotop <dir>` | `GEOtopPY <dir>`, or `run_simulation(dir)` from Python |
| Output location | paths in `geotop.inpts` | the same; `--suffix` / `suffix=` redirects to `output-tabs<suffix>/` without editing the file |
| Output writing | during the run | at the end of the run: an interrupted run leaves no partial output |
| `geotop.log`, `_SUCCESSFUL_RUN` | written | not written; progress on the terminal or through a callback; failure is an exception (CLI: exit code 1) |
| Results in memory | no | one record per internal step, with solver diagnostics |
| Scope | full model | `PointSim=1`, `EnergyBalance=1`, `WaterBalance=0/1` |
| Speed | reference | slower: pure Python, arithmetic kept in GEOtop's order |

Out of scope, and reported with an explicit error rather than a silent
fallback: distributed 2D/3D runs, channel routing, wind-blown snow,
restart/recovery, raster/netCDF outputs, more than one meteo station, more
than one soil type, and slope/aspect/sky view derived from the DEM when their
map is missing.

The example case is `Matsch_P2_Ref_007` (Matsch valley, 1549 m, October 2009,
15-minute step), which the C++ model ran to produce the reference tables
bundled in `tests/1D/`.

## Setup

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from _helpers import (
    compare_with_reference,
    point_df,
    prepare_case,
    reference_dir,
    set_keywords,
)

PY = sys.executable  # for the shell cells

In [ ]:
import geotop_py
from tools.paths import UPSTREAM

print("GEOtopPY", geotop_py.__version__)
print("ported from GEOtop", UPSTREAM.get("branch"), UPSTREAM.get("commit"))

## A case directory, unchanged

`prepare_case` copies the bundled case to `notebooks/runs/` (ignored by git),
leaving out the C++ outputs, so that `tests/1D/` is never written to.

In [ ]:
NAME = "Matsch_P2_Ref_007"
case = prepare_case(NAME)
sorted(p.name for p in case.iterdir())

## Run from the shell

The command takes the case directory, as `geotop` does:

    GEOtopPY <case dir> [--suffix S] [-q]

`--suffix _cli` writes to `output-tabs_cli/` instead of the `output-tabs/`
named in `geotop.inpts`. Below, `python -m geotop_py` is the same command,
called through the notebook's interpreter so that it works without an
activated environment.

In [ ]:
!{PY} -m geotop_py {case} --suffix _cli

## Run from Python

`run_simulation` is the same run. `progress` receives the steps done, the
total and the simulated date; here it prints every 25 %.

In [ ]:
from geotop_py import run_simulation


def progress(done, total, date, _shown=set()):
    q = 4 * done // total
    if q not in _shown:
        _shown.add(q)
        print(f"{done:5d}/{total}  {date:%Y-%m-%d %H:%M}")


t0 = time.perf_counter()
results = run_simulation(str(case), suffix="_py", verbose=False, progress=progress)
print(f"{time.perf_counter() - t0:.1f} s")

## Same files as the C++ model

The file names come from the output keywords in `geotop.inpts`, as in GEOtop.
No `geotop.log` and no `_SUCCESSFUL_RUN` are written.

In [ ]:
py_out = case / "output-tabs_py"
ref_out = reference_dir(NAME)
pd.DataFrame({
    "C++ reference": sorted(p.name for p in ref_out.glob("*.txt")),
    "GEOtopPY": sorted(p.name for p in py_out.glob("*.txt")),
})

In [ ]:
sorted(p.name for p in case.iterdir())

The tables have the same columns, so any reader of GEOtop outputs works.
`point_df` is a thin pandas wrapper used in these notebooks.

In [ ]:
py = point_df(py_out / "point0001.txt")
ref = point_df(ref_out / "point0001.txt")
print(len(py.columns), "columns, identical names:", list(py.columns) == list(ref.columns))
py[["Tair[C]", "Tsurface[C]", "snow_depth[mm]", "snow_water_equivalent[mm]"]].describe()

## Against the C++ outputs

`compare_with_reference` applies the acceptance criterion of
`tools.dashboard` to every table: a value passes when its absolute **or**
relative error is at most 1e-5.

In [ ]:
compare_with_reference(NAME, py_out)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, col in zip(axes, ["snow_depth[mm]", "Tsurface[C]"]):
    ax.plot(ref.index, ref[col], lw=3, alpha=0.4, label="GEOtop C++")
    ax.plot(py.index, py[col], lw=1, label="GEOtopPY")
    ax.set_ylabel(col)
axes[0].legend()
fig.tight_layout()

## Results in memory

`run_simulation` also returns `{point ID: [StepRecord, ...]}`: one record per
internal step the time loop committed, so a halved step gives two records.
The point ID is GEOtop's, the one in the output file names.

`steps_table` flattens the records into rows for pandas: the step (`date`,
`dt`, `trials`), the state and fluxes after it (`swe`, `depth`, `Tg`, ...),
the solver diagnostics (`converged`, `solver_iterations`, `wb_iterations`,
...), the surface diagnostics (`diag_*`) and the forcing (`meteo_*`).
Notebook 03 uses them.

In [ ]:
from geotop_py import steps_table

print("points:", list(results))
steps = pd.DataFrame(steps_table(results[1])).set_index("date")
print(len(steps), "internal steps,", (steps["trials"] > 1).sum(), "needed more than one trial")
steps[["dt", "trials", "swe", "depth", "Tg", "solver_iterations", "wb_iterations"]].head()

## Outside the 1D scope: an explicit error

A configuration the port does not cover fails before the run, with the
reason. From the shell the same message is printed and the exit code is 1.

In [ ]:
dist = prepare_case(NAME, case.parent / "Matsch_distributed")
print(set_keywords(dist, PointSim=0))
try:
    run_simulation(str(dist), verbose=False)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
!{PY} -m geotop_py {dist} -q; echo "exit code $?"

## Speed

On the development machine Matsch_P2 takes about 4 s with GEOtop v3.0
compiled in release mode and about 15 s with GEOtopPY. The port does not
vectorise or reorder the arithmetic, since even last-bit changes alter
solver convergence and snow-layer merges. Long cases (`Bro`, two years at
30 minutes) are better launched from the shell in the background.